### Using the HadISD datset (version 3.4.0.2023f) with PyEarthTools
HadISD is a global sub-daily dataset based on the ISD dataset from NOAA's NCEI. As well as station selection criteria, a suite of quality control tests has been run on the major climatological variables.

The dataset can be downloaded here: https://www.metoffice.gov.uk/hadobs/hadisd/v340_2023f/download.html

In [ ]:
import pyearthtools.pipeline as petpipe
import pyearthtools.data as petdata
import pyearthtools.tutorial


In [ ]:
# train/validation/test split dates
train_start = "1970-01-01T00"
train_end = "2022-12-31T23"

In [ ]:
varname_val_map = {
        "total_cloud_cover": -888., 
        "low_cloud_cover": -999., 
        "mid_cloud_cover": -999.,
        "high_cloud_cover": -999.
    }

In [ ]:
data_prep_pipe = petpipe.Pipeline(
    petdata.archive.hadisd(("010010-99999"), variables = ["total_cloud_cover", "temperatures"]),
    petdata.transforms.values.SetMissingToNaN(varname_val_map),
    # petdata.transforms.variables.Drop(["station_id", "input_station_id"]),
    # petpipe.operations.xarray.conversion.ToNumpy(),
    # petdata.archive.hadisd("010014-99999", variables = ["slp", "dewpoints", "temperatures", "station_id"]),
    # petdata.archive.hadisd(country_code = "24", station_id = "010010", nearest_neighbors = 12, lon= 0.0, lat = 0.0), # Geospatial fence 
    # petdata.archive.hadisd("010014-99999"), # Geospatial fence
)
data_prep_pipe

In [ ]:
ds = data_prep_pipe["1931-01-01T07"]
ds

In some cases you will get the following error when passing a date time to a pipeline object: `IndexWarning: Could not find time in dataset to select on. Petdt('1931-01-01T07')`<br>

This indicates that data for the datetime you chose does not exist. In this case PET will load all data from your station selection.

In [ ]:
# show total_cloud_cover from ds
tcc = ds["total_cloud_cover"]
tcc

In [ ]:
import xarray as xr
from pyearthtools.data.transforms.values import SetMissingToNaN
import numpy as np

def test_set_missing_to_nan():
    data = xr.Dataset({
        "total_cloud_cover": ("time", [0, -999, 50]),
        "low_cloud_cover": ("time", [10, -999, 20]),
    })

    varname_val_map = {
        "total_cloud_cover": -99.0,
        "low_cloud_cover": -999.0,
    }

    transform = SetMissingToNaN(varname_val_map)
    transformed_data = transform.apply(data)

    if np.isnan(transformed_data["total_cloud_cover"].data[1]):
        print("Total cloud cover at index 1 is NaN")
    else:
        print("Total cloud cover at index 1 is not NaN")
    
    if np.isnan(transformed_data["low_cloud_cover"].data[1]):
        print("Low cloud cover at index 1 is NaN")
    
    if transformed_data["total_cloud_cover"].data[2] == 50:
        print("Total cloud cover at index 2 is 50")

In [ ]:
test_set_missing_to_nan()